# RoMa v2 dense matching — any image pair + crop re-detection (Colab T4 / Kaggle 2xT4)

[RoMa v2](https://github.com/Parskatt/RoMaV2) (Edstedt et al., 2025, MIT; DINOv3 ViT-L backbone under the DINOv3 license) predicts a **dense warp A→B + overlap (covisibility)**.
This notebook:
1. clones RoMa v2 and installs only the light dependencies (it keeps the runtime's torch);
2. matches **any pair** of images (URL such as an imgbb `https://i.ibb.co/...` link, a local path, or a Colab upload);
3. **re-detects an object** given a box in A (for example the door handle before the occlusion) in B (for example after the occlusion), in two modes:
   * `fullA+box`: match full A ↔ B, then read the dense warp inside the box and run a RANSAC similarity → map the anchor into B;
   * `crop-only`: match only the crop (template) ↔ B. This is the setting closest to "re-detect from a stored crop";
4. reports the error vs `gt_B` (when given), timing and peak VRAM, and benchmarks the turbo/fast/base/precise settings.

**Runtime:** Colab → *Runtime ▸ Change runtime type ▸ T4 GPU*. Kaggle → *Accelerator GPU T4 x2* + *Internet on* (only `cuda:0` is used).
T4 (sm75) has no native bf16, and RoMa v2 hard-codes bf16 autocast. `AMP_DTYPE="auto"` therefore swaps to fp16 on T4 (set `"bf16"` to compare).
Sim pairs come from `tests/tracking_tools/export_sim_pairs.py <clip> --upload` (only sim renders are uploaded to imgbb); paste its `PAIRS` below.

In [ ]:
# 1) Clone + light install + GPU check
import os, sys, subprocess, time, json, glob, shutil

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out[-2000:])
    return r.returncode

WORK = os.environ.get("ROMA_WORK") or ("/kaggle/working" if os.path.isdir("/kaggle/working")
                                       else "/content" if os.path.isdir("/content") else os.getcwd())
REPO = os.path.join(WORK, "RoMaV2")
OUT_DIR = os.path.join(WORK, "romav2_out")
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.isdir(REPO):
    sh(f"git clone -q https://github.com/Parskatt/RoMaV2 {REPO}")
TRY_FUSED_LOCAL_CORR = False   # optional CUDA kernel (Linux); RoMa falls back to a native-torch local correlation
if not os.environ.get("ROMA_SKIP_INSTALL"):
    # No `pip install -e RoMaV2`: its pyproject pins torchvision>=0.23, which could re-install torch.
    # The runtime's torch is used; src/ goes on sys.path instead.
    sh(f"{sys.executable} -m pip install -q einops rich")
    if TRY_FUSED_LOCAL_CORR:
        sh(f"{sys.executable} -m pip install -q fused-local-corr")
sys.path.insert(0, os.path.join(REPO, "src"))

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| python", sys.version.split()[0])
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {p.name}  {p.total_memory / 2**30:.1f} GB  sm{p.major}{p.minor}")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 4):
    print("WARNING: RoMa v2 + DINOv3 hub code need a recent torch (>=2.4); import will likely fail")

## 2) Settings
`SETTING`: `turbo` (320²), `fast` (512²), `base` (640²), `precise` (800² + 1280² refinement, bidirectional; slowest).

In [ ]:
SETTING = "base"
AMP_DTYPE = "auto"      # "auto": bf16 on sm80+ (A100/L4/H100), fp16 on T4 (sm75), fp32 on CPU | "bf16" | "fp16" | "fp32"
OVERLAP_THR = 0.3       # min RoMa overlap for a box point to vote
BOX_STEP = 2            # grid step (px of A) for the points sampled inside the box
RANSAC_PX = 3.0         # RANSAC reprojection threshold (px of B)
LOCAL_R_PX = 8          # radius (px of A) around the anchor for the local-median estimate
N_HANDLE_LINES = 10     # match lines drawn from the anchor's neighbourhood (thick, red)
N_CONTEXT_LINES = 8     # match lines from the surrounding door/object for global context (thin, blue)
HANDLE_R_FRAC = 0.12    # neighbourhood radius, fraction of the box width (12 px for a 100 px box)
CONTEXT_R_FRAC = (0.2, 0.45)  # context ring around the anchor, fraction of the box width

## 3) Load RoMa v2
The first run downloads the weights (~1.3 GB, GitHub release) and the DINOv3 hub code (architecture only).

In [ ]:
cuda = torch.cuda.is_available()
cap = torch.cuda.get_device_capability(0) if cuda else (0, 0)
amp = AMP_DTYPE if AMP_DTYPE != "auto" else ("fp32" if not cuda else "fp16" if cap[0] < 8 else "bf16")

def patch_amp(dtype):
    """RoMa v2 hard-codes torch.bfloat16 (autocast + frozen DINOv3 weights). T4: swap to fp16;
    fp32 = autocast disabled (slow, NaN-safe fallback). Originals are kept as *.py.orig, so
    switching back is re-running with another dtype (after a runtime restart)."""
    n = 0
    to = {"fp16": "torch.float16", "fp32": "torch.float32"}.get(dtype)
    for py in glob.glob(os.path.join(REPO, "src", "romav2", "**", "*.py"), recursive=True):
        orig = py + ".orig"
        if not os.path.exists(orig):
            shutil.copy(py, orig)
        src = open(orig, encoding="utf-8").read()
        new = src.replace("torch.bfloat16", to) if to else src
        n += new != src
        with open(py, "w", encoding="utf-8") as f:
            f.write(new)
    print(f"AMP dtype = {dtype}  ({n} files patched)")

if "romav2" in sys.modules:
    print(f"romav2 already imported (AMP {amp}); restart the runtime to change AMP_DTYPE")
else:
    patch_amp(amp)
from romav2 import RoMaV2

if "model" not in globals():
    t0 = time.time()
    model = RoMaV2(RoMaV2.Cfg(setting=SETTING))
    print(f"RoMa v2 loaded in {time.time() - t0:.1f}s, {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params"
          + (f", VRAM {torch.cuda.memory_allocated() / 2**20:.0f} MB" if cuda else " (CPU)"))
model.apply_setting(SETTING)

## 4) Image pairs (any pair)
Each entry: `A`, `B` = URL or path. Optional:
* `crop_A` `[x0,y0,x1,y1]`: box in A around the object to re-detect. Default: the whole image, which gives plain pair matching;
* `gt_A` `[x,y]`: the object point in A that is mapped to B. Default: the box centre;
* `gt_B` `[x,y]`: the ground truth of that point in B. When given, the error in px is reported;
* `A_crop_img`: the crop as its own image, used by the `crop-only` mode.

Alternatives: set `MANIFEST` to the `manifest.json` written by `export_sim_pairs.py` (a path or URL; relative image names are resolved next to it), or set `UPLOAD_FROM_DISK=True` on Colab.

In [ ]:
PAIRS = [
    {"name": "toronto (repo sanity)", "A": f"{REPO}/assets/toronto_A.jpg", "B": f"{REPO}/assets/toronto_B.jpg"},
    # Door-handle re-detection (head cam, 256x256): A = last frame before the occlusion, B = after it.
    # Run `export_sim_pairs.py <clip> --upload` and paste its PAIRS, e.g.
    # {"name": "door_head_f44->f75", "A": "https://i.ibb.co/.../door_head_A_f044.png",
    #  "B": "https://i.ibb.co/.../door_head_B_f075.png", "crop_A": [145, 30, 245, 130],
    #  "A_crop_img": "https://i.ibb.co/.../door_head_A_f044_crop100.png", "gt_A": [195.3, 79.7], "gt_B": [210.5, 82.4]},
]
MANIFEST = os.environ.get("ROMA_MANIFEST", "")   # optional path/URL of export_sim_pairs.py's manifest.json
UPLOAD_FROM_DISK = False                         # Colab only: pick 2 local images (A then B)

import io, requests
if MANIFEST:
    if MANIFEST.startswith("http"):
        man = requests.get(MANIFEST, timeout=60).json(); base = MANIFEST.rsplit("/", 1)[0]
    else:
        man = json.load(open(MANIFEST)); base = os.path.dirname(os.path.abspath(MANIFEST))
    def _res(s):
        return s if s.startswith("http") or os.path.isabs(s) else (base + "/" + s)
    for p in man["pairs"]:
        for k in ("A", "B", "A_crop_img"):
            if p.get(k):
                p[k] = _res(p[k])
    PAIRS = man["pairs"]
if UPLOAD_FROM_DISK:
    from google.colab import files
    names = list(files.upload())
    PAIRS = [{"name": "uploaded", "A": names[0], "B": names[1]}]
print(f"{len(PAIRS)} pair(s):", [p["name"] for p in PAIRS])

## 5) Helpers: dense warp lookup and box re-detection

In [ ]:
import numpy as np, cv2
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

_IMG = {}
def load_img(src):
    if src not in _IMG:
        if str(src).startswith("http"):
            r = requests.get(src, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            im = Image.open(io.BytesIO(r.content))
        else:
            im = Image.open(src)
        _IMG[src] = im.convert("RGB")
    return _IMG[src]

def _sync():
    if cuda:
        torch.cuda.synchronize()

def run_match(A, B):
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    _sync(); t0 = time.time()
    preds = model.match(A, B)        # PIL in; the model resizes to the setting's resolution
    _sync(); dt = time.time() - t0
    peak = round(torch.cuda.max_memory_allocated() / 2**20) if cuda else None
    if not torch.isfinite(preds["warp_AB"]).all():
        print("WARNING: non-finite warp (fp16 overflow?) -> try AMP_DTYPE='bf16'")
    return preds, dt, peak

def sample_warp(preds, pts_A, size_A, size_B):
    """Dense warp lookup: A pixels (N,2) -> B pixels (N,2) + overlap (N,). Pixel-centre convention of RoMa."""
    (WA, HA), (WB, HB) = size_A, size_B
    dev = preds["warp_AB"].device
    pts = torch.as_tensor(np.asarray(pts_A, np.float32), device=dev)
    g = torch.stack((pts[:, 0] / WA * 2 - 1, pts[:, 1] / HA * 2 - 1), -1)[None, None]
    w = F.grid_sample(preds["warp_AB"].permute(0, 3, 1, 2).float(), g, align_corners=False)[0, :, 0].T
    ov = F.grid_sample(preds["overlap_AB"].permute(0, 3, 1, 2).float(), g, align_corners=False)[0, 0, 0]
    xb = torch.stack(((w[:, 0] + 1) / 2 * WB, (w[:, 1] + 1) / 2 * HB), -1)
    return xb.cpu().numpy(), ov.cpu().numpy()

def box_grid(box, step):
    xs = np.arange(box[0] + 0.5, box[2], step)
    ys = np.arange(box[1] + 0.5, box[3], step)
    return np.stack(np.meshgrid(xs, ys), -1).reshape(-1, 2)

def locate(pts_A, pts_B, ov, anchor_A):
    """Robust anchor transfer: RANSAC similarity on the confident box correspondences."""
    keep = ov >= OVERLAP_THR
    res = {"n_pts": int(len(ov)), "n_keep": int(keep.sum()), "mean_ov": float(ov.mean()),
           "pred": None, "inl": 0.0, "scale": None, "rot_deg": None}
    if keep.sum() >= 6:
        M, inl = cv2.estimateAffinePartial2D(pts_A[keep].astype(np.float32), pts_B[keep].astype(np.float32),
                                             method=cv2.RANSAC, ransacReprojThreshold=RANSAC_PX)
        if M is not None:
            res.update(pred=M[:, :2] @ anchor_A + M[:, 2], inl=float(inl.mean()),
                       scale=float(np.sqrt(abs(np.linalg.det(M[:, :2])))),
                       rot_deg=float(np.degrees(np.arctan2(M[1, 0], M[0, 0]))))
    return res

def _err(p, gt):
    return None if p is None or gt is None else float(np.hypot(*(np.asarray(p) - np.asarray(gt))))

def local_median(pts_A, pts_B, ov, anchor_A, radius):
    """Anchor transfer from its own neighbourhood: the median displacement of confident points within
    `radius` px of the anchor. A 100 px box mixes the door and the handle, which move differently;
    a single similarity transform over the whole box is pulled toward the dominant (door) motion."""
    near = (np.hypot(*(pts_A - anchor_A).T) <= radius) & (ov >= OVERLAP_THR)
    if near.sum() < 3:
        return None, int(near.sum())
    return anchor_A + np.median(pts_B[near] - pts_A[near], axis=0), int(near.sum())

def redetect(pair):
    """Returns (rows, vis). Mode fullA+box always runs; crop-only runs when pair['A_crop_img'] is set."""
    A, B = load_img(pair["A"]), load_img(pair["B"])
    box = pair.get("crop_A") or [0, 0, A.width, A.height]
    anchor = np.asarray(pair.get("gt_A") or [(box[0] + box[2]) / 2, (box[1] + box[3]) / 2], float)
    gt_B = pair.get("gt_B")
    rows, vis = [], {}
    jobs = [("fullA+box", A, box, anchor)]
    if pair.get("A_crop_img"):
        C = load_img(pair["A_crop_img"])
        s = np.array([C.width / (box[2] - box[0]), C.height / (box[3] - box[1])])
        jobs.append(("crop-only", C, [0, 0, C.width, C.height], (anchor - box[:2]) * s))
    for mode, src, bx, anc in jobs:
        preds, dt, peak = run_match(src, B)
        pA = box_grid(bx, BOX_STEP * (src.width / A.width if mode == "crop-only" else 1))
        pB, ov = sample_warp(preds, pA, src.size, B.size)
        r = locate(pA, pB, ov, anc)
        sA = src.width / A.width if mode == "crop-only" else 1.0
        loc, n_loc = local_median(pA, pB, ov, anc, LOCAL_R_PX * sA)
        d, dov = sample_warp(preds, anc[None], src.size, B.size)
        rows.append({"pair": pair["name"], "mode": mode, "setting": SETTING, "amp": amp,
                     "time_s": round(dt, 3), "peak_MB": peak, "n_keep": r["n_keep"], "n_pts": r["n_pts"],
                     "mean_ov": round(r["mean_ov"], 3), "inlier": round(r["inl"], 3),
                     "scale": None if r["scale"] is None else round(r["scale"], 3),
                     "rot_deg": None if r["rot_deg"] is None else round(r["rot_deg"], 1),
                     "pred_ransac": None if r["pred"] is None else [round(float(v), 1) for v in r["pred"]],
                     "err_ransac_px": None if _err(r["pred"], gt_B) is None else round(_err(r["pred"], gt_B), 1),
                     "pred_local": None if loc is None else [round(float(v), 1) for v in loc], "n_local": n_loc,
                     "err_local_px": None if _err(loc, gt_B) is None else round(_err(loc, gt_B), 1),
                     "pred_direct": [round(float(v), 1) for v in d[0]], "ov_direct": round(float(dov[0]), 3),
                     "err_direct_px": None if _err(d[0], gt_B) is None else round(_err(d[0], gt_B), 1)})
        vis[mode] = dict(src=src, B=B, preds=preds, box=bx, anchor=anc, pA=pA, pB=pB, ov=ov,
                         pred=r["pred"], local=loc, direct=d[0], gt_B=gt_B)
    return rows, vis

def print_rows(rows, cols=("pair", "mode", "setting", "amp", "time_s", "peak_MB", "n_keep", "mean_ov", "inlier",
                           "scale", "err_ransac_px", "err_local_px", "err_direct_px", "ov_direct", "pred_direct")):
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows)[list(cols)])
    except Exception:
        print(" | ".join(cols))
        for r in rows:
            print(" | ".join(str(r.get(c)) for c in cols))

## 6) Visualization
Per mode, five panels:
1. A (or the crop) with the box and anchor;
2. B with every box point mapped by the warp, coloured by overlap. This is RoMa's "where is the crop in B" map, the counterpart of the NCC similarity map. Markers: RANSAC similarity over the whole box (red x), local median within `LOCAL_R_PX` of the anchor (magenta x), direct warp of the anchor pixel (orange +), GT (lime circle);
3. the overlap map of A (does this A pixel appear in B?);
4. B warped into A's frame and blended to white where the overlap is low;
5. match lines, with A's box zoomed on the left and B on the right. Thick red lines come from pixels near the anchor (the handle). Thin blue lines come from a ring around it (the door), for global context. The orange line is the anchor's own direct warp. Correct when it ends in the green GT ring.

In [ ]:
def warp_B_into_A(vis):
    preds, B = vis["preds"], vis["B"]
    h, w = preds["warp_AB"].shape[1:3]
    dev = preds["warp_AB"].device
    xb = torch.from_numpy(np.asarray(B.resize((w, h)))).float().div(255).permute(2, 0, 1)[None].to(dev)
    out = F.grid_sample(xb, preds["warp_AB"].float(), align_corners=False)[0]
    ov = preds["overlap_AB"][0, ..., 0].float()
    out = ov * out + (1 - ov)
    return out.permute(1, 2, 0).clamp(0, 1).cpu().numpy()

def spread_sample(pts, n, seed=0):
    """Indices of n points spread over pts (farthest-point sampling)."""
    if len(pts) <= n:
        return np.arange(len(pts))
    sel = [int(np.random.default_rng(seed).integers(len(pts)))]
    d = np.hypot(*(pts - pts[sel[0]]).T)
    for _ in range(n - 1):
        i = int(np.argmax(d)); sel.append(i)
        d = np.minimum(d, np.hypot(*(pts - pts[i]).T))
    return np.array(sel)

def match_lines(vis):
    """(handle_idx, context_idx): confident matches near the anchor, and from a ring around it (the door).
    Box corners are often floor/sky, so context is taken from the ring with a stricter overlap."""
    bw = vis["box"][2] - vis["box"][0]
    dist = np.hypot(*(vis["pA"] - vis["anchor"]).T)
    conf = vis["ov"] >= OVERLAP_THR
    hi = np.where((dist <= HANDLE_R_FRAC * bw) & conf)[0]
    ci = np.where((dist >= CONTEXT_R_FRAC[0] * bw) & (dist <= CONTEXT_R_FRAC[1] * bw) & (vis["ov"] >= 0.6))[0]
    if len(ci) < N_CONTEXT_LINES:
        ci = np.where((dist > HANDLE_R_FRAC * bw) & conf)[0]
    hsel = hi[spread_sample(vis["pA"][hi], N_HANDLE_LINES)] if len(hi) else hi
    csel = ci[spread_sample(vis["pA"][ci], N_CONTEXT_LINES)] if len(ci) else ci
    return hsel, csel

def draw_match_lines(ax, v, src, B):
    x0, y0, x1, y1 = [int(round(c)) for c in v["box"]]
    crop = src[y0:y1, x0:x1]
    s = B.shape[0] / crop.shape[0]
    crop = np.asarray(Image.fromarray(crop).resize((int(round(crop.shape[1] * s)), B.shape[0]), Image.NEAREST))
    gap = 12
    xoff = crop.shape[1] + gap
    canvas = np.full((B.shape[0], xoff + B.shape[1], 3), 255, np.uint8)
    canvas[:, :crop.shape[1]] = crop; canvas[:, xoff:] = B
    ax.imshow(canvas)
    a_xy = lambda p: (np.asarray(p, float) - [x0, y0]) * s
    b_xy = lambda p: np.asarray(p, float) + [xoff, 0]
    hsel, csel = match_lines(v)
    for idx, col, lw, ms in ((csel, "deepskyblue", 0.9, 3), (hsel, "orangered", 1.6, 4)):
        for i in idx:
            a, b = a_xy(v["pA"][i]), b_xy(v["pB"][i])
            ax.plot([a[0], b[0]], [a[1], b[1]], "-", color=col, lw=lw, alpha=0.85)
            ax.plot(*a, "o", color=col, ms=ms); ax.plot(*b, "o", color=col, ms=ms)
    a, b = a_xy(v["anchor"]), b_xy(v["direct"])
    ax.plot([a[0], b[0]], [a[1], b[1]], "-", color="white", lw=4.0)
    ax.plot([a[0], b[0]], [a[1], b[1]], "-", color="orange", lw=2.2)
    ax.plot(*a, "+", color="orange", ms=14, mew=2.5); ax.plot(*b, "+", color="orange", ms=14, mew=2.5)
    if v["gt_B"] is not None:
        ax.plot(*b_xy(v["gt_B"]), "o", mfc="none", mec="lime", ms=16, mew=2)
    ax.set_title(f"A box (zoomed) | B: {len(hsel)} handle (red), {len(csel)} door context (blue), anchor (orange)",
                 fontsize=9)

def show(pair, vis, save=True):
    for mode, v in vis.items():
        fig, ax = plt.subplots(1, 5, figsize=(25, 5.2))
        src, B = np.asarray(v["src"]), np.asarray(v["B"])
        x0, y0, x1, y1 = v["box"]
        ax[0].imshow(src); ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="cyan", lw=1.5))
        ax[0].plot(*v["anchor"], "r+", ms=14, mew=2); ax[0].set_title(f"{mode}: A / crop + anchor")
        ax[1].imshow(B)
        sc = ax[1].scatter(v["pB"][:, 0], v["pB"][:, 1], c=v["ov"], s=4, cmap="viridis", vmin=0, vmax=1, alpha=0.8)
        plt.colorbar(sc, ax=ax[1], fraction=0.046, label="overlap")
        if v["pred"] is not None:
            ax[1].plot(*v["pred"], "rx", ms=14, mew=3, label="RANSAC (box)")
        if v["local"] is not None:
            ax[1].plot(*v["local"], "x", color="magenta", ms=12, mew=2, label="local median")
        ax[1].plot(*v["direct"], "+", color="orange", ms=14, mew=2, label="direct warp")
        if v["gt_B"] is not None:
            ax[1].plot(*v["gt_B"], "o", mfc="none", mec="lime", ms=16, mew=2, label="GT")
        ax[1].set_xlim(0, B.shape[1]); ax[1].set_ylim(B.shape[0], 0); ax[1].legend(loc="lower right", fontsize=8)
        ax[1].set_title("B: box points mapped, colour = overlap")
        ax[2].imshow(src, alpha=0.35)
        ov = v["preds"]["overlap_AB"][0, ..., 0].float().cpu().numpy()
        im = ax[2].imshow(ov, cmap="magma", vmin=0, vmax=1, alpha=0.75, extent=(0, src.shape[1], src.shape[0], 0))
        plt.colorbar(im, ax=ax[2], fraction=0.046); ax[2].set_title("overlap_AB over A")
        ax[3].imshow(warp_B_into_A(v)); ax[3].set_title("B warped into A (white = no overlap)")
        draw_match_lines(ax[4], v, src, B)
        for a in ax:
            a.axis("off")
        fig.suptitle(f"{pair['name']}  [{mode}, {SETTING}, {amp}]")
        fig.tight_layout()
        if save:
            fn = os.path.join(OUT_DIR, f"{pair['name'].replace('>', '').replace('/', '_').replace(' ', '_')}_{mode}_{SETTING}.png")
            fig.savefig(fn, dpi=110)
        plt.show()

## 7) Run all pairs

In [ ]:
RESULTS = []
for pair in PAIRS:
    rows, vis = redetect(pair)
    RESULTS += rows
    show(pair, vis)
print_rows(RESULTS)
with open(os.path.join(OUT_DIR, f"results_{SETTING}.json"), "w") as f:
    json.dump(RESULTS, f, indent=1)

## 8) Benchmark: settings vs time / VRAM / error
Each setting gets one warm-up match, then each pair is re-run. `precise` is bidirectional and adds the 1280² refinement.

In [ ]:
BENCH = ["turbo", "fast", "base", "precise"]
BENCH_ROWS = []
_keep = SETTING
for s in BENCH:
    SETTING = s
    model.apply_setting(s)
    run_match(load_img(PAIRS[0]["A"]), load_img(PAIRS[0]["B"]))   # warm-up
    for pair in PAIRS:
        BENCH_ROWS += redetect(pair)[0]
SETTING = _keep
model.apply_setting(SETTING)
print_rows(BENCH_ROWS)
with open(os.path.join(OUT_DIR, "bench.json"), "w") as f:
    json.dump(BENCH_ROWS, f, indent=1)

In [ ]:
# 9) Download the figures and JSON
archive = shutil.make_archive(os.path.join(WORK, "romav2_out"), "zip", OUT_DIR)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass   # Kaggle: grab it from /kaggle/working in the Output tab

### Reading the results
* Head camera ≈ 6.4 mm/px at 1.4 m: **< 5 px ≈ < 3 cm** is a usable re-detection, enough to re-seed the tracker near the handle.
* `inlier` (RANSAC inlier fraction) and `mean_ov` are the confidence signals to gate a re-seed on. A wrong-but-confident result (the twin lever, the door edge) is the failure mode that matters, so compare them across pairs.
* `crop-only` is the "re-detect from a stored crop" setting; `fullA+box` uses the whole pre-occlusion frame as context.
* NCC baseline on the same clip (crop 100): the truth ranked 10–18th with score 0.36–0.54 vs. false positives at 0.56–0.70 (see `docs/plans/3d_tracking.md` §13.11).